# 04 — NVS Internal Trade Data EDA (Stage 4)

**Run locally. No data leaves your machine.**

Known data properties:
- CSV format
- ISO-2 two-letter country codes
- Shipping lane columns: origin + destination country
- Two static timestamps: 2025-11-03 and 2025-11-13
- M/X (import/export) NOT labeled — inferred from NVS production countries
- No HS codes — crosswalk applied via industry group

**Update `NVS_FILE_PATH` in Cell 1 before running.**

In [ ]:
# ── Cell 0: Imports & Config ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 4)

# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
NVS_FILE_PATH = 'path/to/your/nvs_data.csv'   # <-- change this
OUTPUT_DIR = Path('outputs/nvs_eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Stage 3 locked corridors (canonical sorted pairs) ────────────────────────
STAGE3_CORRIDORS = {
    frozenset(['USA','CHN']), frozenset(['USA','DEU']), frozenset(['USA','IND']),
    frozenset(['USA','BRA']), frozenset(['USA','CAN']), frozenset(['USA','MEX']),
    frozenset(['CHN','DEU']), frozenset(['CHN','IND']), frozenset(['CHN','JPN']),
    frozenset(['CHN','KOR']), frozenset(['DEU','IND']), frozenset(['DEU','BRA']),
    frozenset(['IND','BRA']),
}
STAGE3_KEYS = {'_'.join(sorted(list(p))) for p in STAGE3_CORRIDORS}

# ── ISO-2 → ISO-3 ─────────────────────────────────────────────────────────────
ISO2_TO_ISO3 = {
    'US':'USA','CN':'CHN','DE':'DEU','IN':'IND','BR':'BRA',
    'CA':'CAN','MX':'MEX','JP':'JPN','KR':'KOR','FR':'FRA',
    'DK':'DNK','GB':'GBR','AU':'AUS','ZA':'ZAF','RU':'RUS',
    'IT':'ITA','ES':'ESP','NL':'NLD','SE':'SWE','NO':'NOR',
    'PL':'POL','CZ':'CZE','AR':'ARG','CL':'CHL','CO':'COL',
    'EG':'EGY','NG':'NGA','KE':'KEN','TH':'THA','ID':'IDN',
    'MY':'MYS','PH':'PHL','VN':'VNM','PK':'PAK','BD':'BGD',
    'TR':'TUR','SA':'SAU','AE':'ARE','IL':'ISR','SG':'SGP',
}

# NVS production countries (ISO-2) — used for M/X inference
NVS_PRODUCTION_ISO2 = {'DK','US','CA','BR','IN','CN','FR','DE','CZ'}

SNAPSHOT_DATES = ['2025-11-03', '2025-11-13']

def iso2_to_iso3(series):
    return series.astype(str).str.strip().str.upper().map(
        lambda x: ISO2_TO_ISO3.get(x, x)
    )

print('✓ Config loaded.')

In [ ]:
# ── Cell 1: Load Data ─────────────────────────────────────────────────────────
df = pd.read_csv(NVS_FILE_PATH, low_memory=False)
print(f'✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
df.head(3)

## Section 1 — Schema Audit
Column names, dtypes, null counts, unique value counts.

In [ ]:
# ── Cell 2: Schema Audit ──────────────────────────────────────────────────────
schema = pd.DataFrame({
    'dtype':   df.dtypes.astype(str),
    'nulls':   df.isnull().sum(),
    'null_%':  (df.isnull().mean() * 100).round(1),
    'unique':  df.nunique(),
    'sample':  [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
})
print('Column inventory:')
display(schema)

## Section 2 — Timestamp Analysis
Identify the two date columns and understand the relationship between them.

In [ ]:
# ── Cell 3: Detect & Parse Date Columns ──────────────────────────────────────
date_keywords = ['date','time','period','year','month','scenario']
date_cols = [c for c in df.columns if any(k in c.lower() for k in date_keywords)]
print(f'Detected date-like columns: {date_cols}')

parsed = {}
for col in date_cols:
    s = pd.to_datetime(df[col], errors='coerce')
    if s.notna().sum() > 0:
        parsed[col] = s
        print(f'\n[{col}]')
        print(f'  Parsed: {s.notna().sum():,}/{len(df):,}  |  Min: {s.min().date()}  Max: {s.max().date()}')
        print(f'  Unique dates: {s.nunique()}')
        print(f'  By year:'); print(s.dt.year.value_counts().sort_index().to_string())

In [ ]:
# ── Cell 4: Gap Between the Two Timestamps ───────────────────────────────────
non_scenario = [c for c in parsed if 'scenario' not in c.lower()]
scenario_keys = [c for c in parsed if 'scenario' in c.lower()]

if non_scenario and scenario_keys:
    base_col = non_scenario[0]
    scen_col = scenario_keys[0]
    delta = (parsed[scen_col] - parsed[base_col]).dt.days
    print(f'Gap: [{scen_col}] − [{base_col}]  (days)')
    print(f'  Mean: {delta.mean():.0f}  |  Median: {delta.median():.0f}  |  Min: {delta.min():.0f}  |  Max: {delta.max():.0f}')
    print(f'  Negative gaps (scenario before base): {(delta < 0).sum():,}')

    fig, ax = plt.subplots(figsize=(8,3))
    delta.dropna().hist(bins=40, ax=ax, color='#2c7bb6', edgecolor='white')
    ax.set_title(f'Gap distribution: {scen_col} − {base_col} (days)')
    ax.set_xlabel('Days'); ax.set_ylabel('Rows')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'timestamp_gap.png', dpi=120); plt.show()
else:
    print('Could not find two distinct timestamp columns. Adjust detection above.')

## Section 2b — Snapshot Delta Analysis

> ⚠️ **NVS data has only 2 static timestamps (Nov 3 and Nov 13, 2025). This is NOT a time series — it is two scenario snapshots.**
>
> **Implication:** Cannot train a time-series model on 2 timestamps.  
> NVS data is used as **impact quantification**, not training labels.  
> Delta (B − A) per corridor/BF/industry = revenue-at-risk per tariff event.  
> The GDELT/ICEWS `features_master.csv` remains the training backbone.

In [ ]:
# ── Cell 5: Snapshot Split & Delta ───────────────────────────────────────────
# !! UPDATE these column names if auto-detection is wrong
TIMESTAMP_COL = date_cols[0] if date_cols else None   # <-- adjust if needed

if TIMESTAMP_COL:
    ts = pd.to_datetime(df[TIMESTAMP_COL], errors='coerce')
    unique_dates = sorted(ts.dropna().unique())
    print(f'Unique dates in [{TIMESTAMP_COL}]: {[str(pd.Timestamp(d).date()) for d in unique_dates]}')

    if len(unique_dates) == 2:
        d1, d2 = unique_dates
        snap_a = df[ts == d1].copy()
        snap_b = df[ts == d2].copy()
        print(f'\nSnapshot A ({pd.Timestamp(d1).date()}): {len(snap_a):,} rows')
        print(f'Snapshot B ({pd.Timestamp(d2).date()}): {len(snap_b):,} rows')
    else:
        print(f'Expected 2 dates, found {len(unique_dates)}. Adjust TIMESTAMP_COL.')
        snap_a, snap_b = None, None
else:
    print('No timestamp column found.')
    snap_a, snap_b = None, None

In [ ]:
# ── Cell 6: Delta by Value Column ─────────────────────────────────────────────
# !! UPDATE these if auto-detection misses your columns
val_keywords = ['value','revenue','amount','volume','qty','quantity','usd','eur','dkk','sales']
VALUE_COLS = [c for c in df.columns if any(k in c.lower() for k in val_keywords)]
print(f'Value/volume columns detected: {VALUE_COLS}')

if snap_a is not None and VALUE_COLS:
    for vcol in VALUE_COLS:
        va = pd.to_numeric(snap_a[vcol], errors='coerce').sum()
        vb = pd.to_numeric(snap_b[vcol], errors='coerce').sum()
        pct = 100*(vb-va)/va if va != 0 else float('nan')
        print(f'\n[{vcol}]  A={va:,.0f}  B={vb:,.0f}  Δ={vb-va:+,.0f}  ({pct:+.1f}%)')

In [ ]:
# ── Cell 7: Delta by Industry Group ──────────────────────────────────────────
# !! UPDATE IND_COL if wrong
ind_keywords = ['industry','sector','group','category','vertical']
IND_COL = next((c for c in df.columns if any(k in c.lower() for k in ind_keywords)), None)
print(f'Industry group column: {IND_COL}')

if snap_a is not None and IND_COL and VALUE_COLS:
    primary_val = VALUE_COLS[0]
    g_a = snap_a.groupby(IND_COL)[primary_val].apply(lambda x: pd.to_numeric(x, errors='coerce').sum())
    g_b = snap_b.groupby(IND_COL)[primary_val].apply(lambda x: pd.to_numeric(x, errors='coerce').sum())
    delta_df = pd.DataFrame({'Snapshot_A': g_a, 'Snapshot_B': g_b}).fillna(0)
    delta_df['Delta'] = delta_df['Snapshot_B'] - delta_df['Snapshot_A']
    delta_df['Delta_%'] = (100 * delta_df['Delta'] / delta_df['Snapshot_A'].replace(0, float('nan'))).round(1)
    delta_df = delta_df.sort_values('Delta_%')
    print(f'\nDelta by Industry Group [{primary_val}]:')
    display(delta_df)

    # Bar chart
    fig, ax = plt.subplots(figsize=(10,4))
    colors = ['#d7191c' if v < 0 else '#2c7bb6' for v in delta_df['Delta_%']]
    delta_df['Delta_%'].plot(kind='barh', ax=ax, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'% Change in {primary_val} by Industry Group (Nov 3 → Nov 13)')
    ax.set_xlabel('% change')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'delta_by_industry.png', dpi=120); plt.show()

## Section 3 — Scenario Date Column
Check if the scenario date is sparse enough to be a binary event label.

In [ ]:
# ── Cell 8: Scenario Date Analysis ───────────────────────────────────────────
SCENARIO_COL = next((c for c in df.columns if 'scenario' in c.lower()), None)
print(f'Scenario column: {SCENARIO_COL}')

if SCENARIO_COL:
    s = pd.to_datetime(df[SCENARIO_COL], errors='coerce')
    print(f'Unique scenario dates: {s.nunique()}  |  Nulls: {s.isna().sum():,}')
    print(f'Sparsity: {s.nunique()} unique / {len(df):,} rows = {s.nunique()/len(df)*100:.2f}%')
    print()
    for d in sorted(s.dropna().unique()):
        print(f'  {pd.Timestamp(d).date()}  →  {(s == d).sum():,} rows')

    if s.nunique() / len(df) < 0.05:
        print('\n→ Low cardinality. Likely a tariff-event label column. Candidate for binary y.')
    else:
        print('\n→ High cardinality. Likely a continuous timestamp, not a discrete event marker.')

## Section 4 — Shipping Lanes, Corridor Coverage & M/X Inference

In [ ]:
# ── Cell 9: Detect Lane Columns ───────────────────────────────────────────────
origin_keywords = ['origin','source','from','shipper','exporter','sender']
dest_keywords   = ['dest','destination','to','consignee','importer','receiver','target']

ORIGIN_COL = next((c for c in df.columns if any(k in c.lower() for k in origin_keywords)), None)
DEST_COL   = next((c for c in df.columns if any(k in c.lower() for k in dest_keywords)), None)

print(f'Origin column : {ORIGIN_COL}')
print(f'Destination   : {DEST_COL}')
print()

if ORIGIN_COL and DEST_COL:
    orig_iso2 = df[ORIGIN_COL].astype(str).str.strip().str.upper()
    dest_iso2 = df[DEST_COL].astype(str).str.strip().str.upper()
    orig_iso3 = iso2_to_iso3(df[ORIGIN_COL])
    dest_iso3 = iso2_to_iso3(df[DEST_COL])

    all_countries = set(orig_iso3.dropna()) | set(dest_iso3.dropna())
    print(f'Unique countries across lanes ({len(all_countries)}): {sorted(all_countries)}')

In [ ]:
# ── Cell 10: M/X Inference from Production Countries ─────────────────────────
if ORIGIN_COL and DEST_COL:
    is_export  = orig_iso2.isin(NVS_PRODUCTION_ISO2)
    is_import  = dest_iso2.isin(NVS_PRODUCTION_ISO2)
    is_both    = is_export & is_import
    is_neither = ~is_export & ~is_import

    print('M/X inference (origin/dest vs NVS production countries):')
    print(f'  Export (X) rows  : {is_export.sum():,}')
    print(f'  Import (M) rows  : {is_import.sum():,}')
    print(f'  Both (intra-NVS) : {is_both.sum():,}')
    print(f'  Neither          : {is_neither.sum():,}')

    # Add flow column to df for downstream use
    df['inferred_flow'] = 'unknown'
    df.loc[is_export & ~is_import, 'inferred_flow'] = 'X'
    df.loc[is_import & ~is_export, 'inferred_flow'] = 'M'
    df.loc[is_both,   'inferred_flow'] = 'intra_NVS'

    print()
    print('Value breakdown by inferred flow:')
    if VALUE_COLS:
        pv = VALUE_COLS[0]
        print(df.groupby('inferred_flow')[pv].apply(
            lambda x: pd.to_numeric(x, errors='coerce').sum()
        ).sort_values(ascending=False).apply(lambda v: f'{v:,.0f}').to_string())

In [ ]:
# ── Cell 11: Corridor Coverage vs Stage 3 ────────────────────────────────────
if ORIGIN_COL and DEST_COL:
    corridor_series = pd.Series(
        ['_'.join(sorted([o, d])) for o, d in zip(orig_iso3.fillna('UNK'), dest_iso3.fillna('UNK'))]
    )

    nvs_corridors = set(corridor_series.unique())
    matched  = nvs_corridors & STAGE3_KEYS
    missing  = STAGE3_KEYS - nvs_corridors
    extra    = nvs_corridors - STAGE3_KEYS

    print(f'Unique bilateral corridors in NVS data : {len(nvs_corridors)}')
    print(f'✓ Matched to Stage 3 corridors         : {len(matched)}  {sorted(matched)}')
    print(f'✗ Stage 3 corridors absent from NVS    : {len(missing)}  {sorted(missing)}')
    print(f'+ Extra NVS corridors not in Stage 3   : {len(extra)}')

    print(f'\nTop 20 corridors by row count:')
    print(corridor_series.value_counts().head(20).to_string())

    # Store for join
    df['corridor'] = corridor_series

## Section 5 — Business Function & Industry Group

In [ ]:
# ── Cell 12: BF & Industry Group Distributions ────────────────────────────────
bf_keywords  = ['business','bf','function','segment']
BF_COL  = next((c for c in df.columns if any(k in c.lower() for k in bf_keywords)), None)
# IND_COL already detected in Cell 7

for label, col in [('Business Function', BF_COL), ('Industry Group', IND_COL)]:
    if col:
        print(f'\n[{label}]  column: "{col}"')
        display(df[col].value_counts(dropna=False).to_frame())
    else:
        print(f'\n⚠  [{label}] not detected.')

In [ ]:
# ── Cell 13: BF × Industry Cross-tab + Heatmap ───────────────────────────────
if BF_COL and IND_COL:
    ct = pd.crosstab(df[BF_COL], df[IND_COL])
    display(ct)

    fig, ax = plt.subplots(figsize=(13, 4))
    im = ax.imshow(ct.values, aspect='auto', cmap='Blues')
    ax.set_xticks(range(len(ct.columns)))
    ax.set_yticks(range(len(ct.index)))
    ax.set_xticklabels(ct.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(ct.index, fontsize=9)
    ax.set_title('Row counts: Business Function × Industry Group')
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'bf_industry_heatmap.png', dpi=120); plt.show()

## Section 6 — Trade Value & Volume Distributions

In [ ]:
# ── Cell 14: Value Column Stats ───────────────────────────────────────────────
if VALUE_COLS:
    stats = []
    for col in VALUE_COLS:
        s = pd.to_numeric(df[col], errors='coerce')
        stats.append({
            'column': col,
            'non_null': s.notna().sum(),
            'zero_%': round(100*(s==0).sum()/len(s), 1),
            'negative': (s<0).sum(),
            'mean': s.mean(),
            'median': s.median(),
            'p95': s.quantile(0.95),
            'max': s.max(),
        })
    display(pd.DataFrame(stats).set_index('column'))
else:
    print('⚠  No value columns detected. Update val_keywords or VALUE_COLS above.')

In [ ]:
# ── Cell 15: Value Distributions (log scale) ──────────────────────────────────
if VALUE_COLS:
    n = len(VALUE_COLS)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 4))
    if n == 1: axes = [axes]
    for ax, col in zip(axes, VALUE_COLS):
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        s_pos = s[s > 0]
        if len(s_pos):
            np.log1p(s_pos).hist(bins=50, ax=ax, color='#2c7bb6', edgecolor='white')
            ax.set_title(f'log(1 + {col})')
        ax.set_xlabel('log(1+value)'); ax.set_ylabel('Count')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'value_distributions.png', dpi=120); plt.show()

In [ ]:
# ── Cell 16: Value by BF and Industry Group ───────────────────────────────────
if VALUE_COLS:
    pv = VALUE_COLS[0]
    for label, col in [('Business Function', BF_COL), ('Industry Group', IND_COL)]:
        if col:
            print(f'\n[{pv}] total by {label}:')
            totals = df.groupby(col)[pv].apply(
                lambda x: pd.to_numeric(x, errors='coerce').sum()
            ).sort_values(ascending=False)
            display(totals.apply(lambda v: f'{v:,.0f}').to_frame(name='total'))

## Section 7 — Missing Data

In [ ]:
# ── Cell 17: Missing Data ──────────────────────────────────────────────────────
missing_pct = df.isnull().mean().sort_values(ascending=False)
cols_with_missing = missing_pct[missing_pct > 0]

if cols_with_missing.empty:
    print('✓ No missing values.')
else:
    display(cols_with_missing.apply(lambda x: f'{100*x:.1f}%').to_frame(name='missing_%'))
    fig, ax = plt.subplots(figsize=(10, max(3, len(cols_with_missing)*0.4)))
    cols_with_missing.plot(kind='barh', ax=ax, color='#d7191c')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title('Missing data by column')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'missing_data.png', dpi=120); plt.show()

## Section 8 — Stage 4 Join Readiness

In [ ]:
# ── Cell 18: Join Readiness Checklist ────────────────────────────────────────
print('Required join key for features_master.csv: (corridor, week)')
print('No HS codes in NVS → crosswalk via industry group\n')

checks = [
    ('Origin column',       ORIGIN_COL,                         'corridor mapping'),
    ('Destination column',  DEST_COL,                           'corridor mapping'),
    ('Timestamp column',    date_cols[0] if date_cols else None, 'snapshot split'),
    ('Scenario date col',   SCENARIO_COL,                       'y-label candidate'),
    ('Business Function',   BF_COL,                             'revenue weighting'),
    ('Industry Group',      IND_COL,                            'HS crosswalk'),
    ('Value/Volume col',    VALUE_COLS[0] if VALUE_COLS else None, 'impact calculation'),
]

all_ok = True
for name, col, note in checks:
    status = '✓' if col else '✗'
    if not col: all_ok = False
    print(f'  {status}  {name:<28} → {str(col):<30} ({note})')

print()
print('✓ All present — ready for Stage 4 join.' if all_ok else '✗ Resolve missing columns before join.')

print('\nHS crosswalk (Industry Group → HS chapters):')
crosswalk = {
    'Dairy':                       ['3507','2102','3002'],
    'Animal':                      ['2309','3507'],
    'Food & Beverages':            ['3507','2102','2106','3002'],
    'Dietary Supplements':         ['3002','3504','2936'],
    'Advanced Health & Nutrition': ['3002','3504','2936','2106'],
    'Specialized Nutrition':       ['3504','2936','3002'],
    'Bioenergy':                   ['3507','3821'],
    'Household Care':              ['3507','3824'],
    'Tech':                        ['3822','3824'],
    'Plant':                       ['3002','3507','3821'],
}
for ig, hs in crosswalk.items():
    print(f'  {ig:<35} → HS {", ".join(hs)}')

## Summary — Key Questions to Answer After Running

Share these answers to proceed to Stage 4 join logic:

1. **Timestamp columns** — what are the exact column names, and what does the gap distribution look like?
2. **Scenario date** — is it sparse (<5% unique)? If yes → use as binary tariff-event label
3. **Corridor coverage** — which Stage 3 corridors are missing from NVS?
4. **M/X balance** — how many rows are Export vs Import after inference?
5. **Primary value column** — revenue, volume, or both? Which currency?
6. **What tariff event** does the Nov 3 → Nov 13 delta represent?